# LiPAD — Corrosion Detection Training

Train a **YOLO segmentation** corrosion detector with Roboflow-aligned settings.

## Quick start (Google Colab)

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Upload or clone `LIPAD_YOLO_TRAINING` into `/content/LIPAD_YOLO_TRAINING`.
3. Put raw images/labels under `corrosion_detection/datasets_raw/` **or** directly in `corrosion_detection/datasets/images/{train,val}` with matching `labels/{train,val}`.
4. Run all cells top to bottom.
5. Best weights are saved under `corrosion_detection/runs/<model>/corrosion_<model>_seg/weights/best.pt`.

## Configuration applied

| Setting | Value |
|---|---|
| Epochs | 100 |
| Learning rate | 0.01 |
| Optimizer | SGD |
| Image size | 640×640 (stretch) |
| Auto-orient | Yes |
| Contrast | Adaptive equalization (CLAHE) |
| Classes | fair, poor, severe (2 remapped, 3 dropped) |
| Aug copies (train) | 3 per source image (offline photometric) |
| Flips | Horizontal + vertical |
| Rotation | ±15° |
| Exposure | ±25% |
| Blur | up to 2.5 px |
| Noise | up to 10% of pixels |

Edit defaults in `shared/corrosion_config.py` if your Roboflow export uses different class ids.

In [1]:
%pip install torch torchvision
%pip install ultralytics pyyaml albumentations opencv-python-headless pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.4 MB/s eta 0:00:0000:01


In [2]:
# @title 1. Environment setup
import sys
from pathlib import Path

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content
    # After cloning/uploading, point here if needed:
    # %cd /content/LIPAD_YOLO_TRAINING
else:
    ROOT = Path(r'C:/Users/Admin/PROJECT_LIPAD/Corrosion/LIPAD_YOLO_TRAINING')
    if (Path.cwd() / 'shared').exists():
        ROOT = Path.cwd()
    elif (Path.cwd().parent / 'shared').exists():
        ROOT = Path.cwd().parent
    %cd {ROOT}
    sys.path.insert(0, str(ROOT))
    print('Repo root:', ROOT)


Mounted at /content/drive
/content


In [ ]:
import os
import sys

# Clone repository if not already cloned
!git clone https://github.com/sarieljandaniel-svg/Corrosion.git

# Automatically find where 'corrosion_detection' is located inside /content and change directory to it
for root, dirs, files in os.walk('/content/Corrosion'):
    if 'corrosion_detection' in dirs:
        target_path = os.path.join(root, 'corrosion_detection')
        os.chdir(target_path)
        sys.path.append(target_path)
        print(f"Successfully changed directory to: {target_path}")
        break

# Now try your import
from shared.corrosion_config import config_summary, CorrosionPreprocessConfig, CorrosionTrainConfig

print(config_summary())
print('\nPreprocess config:', CorrosionPreprocessConfig())
print('Train config     :', CorrosionTrainConfig())

fatal: destination path 'Corrosion' already exists and is not an empty directory.


FileNotFoundError: [Errno 2] No such file or directory: '/content/Corrosion/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection'

In [5]:
# @title 2. Review training configuration
import sys
import os

# Add the root directory of your project to the Python path
# Replace or adjust the path to match where your project folder is located in Colab/Cursor
sys.path.append(os.path.abspath('/content/LIPAD_YOLO_TRAINING')) 
# Alternatively, if running relative to your workspace:
# sys.path.append(os.path.abspath('.'))

from shared.corrosion_config import config_summary, CorrosionPreprocessConfig, CorrosionTrainConfig

print(config_summary())
print('\nPreprocess config:', CorrosionPreprocessConfig())
print('Train config     :', CorrosionTrainConfig())

ModuleNotFoundError: No module named 'shared'

In [ ]:
# @title 3. Prepare dataset folders
from shared.paths import ensure_dataset_layout, datasets_dir, task_root

ensure_dataset_layout('corrosion_detection')
raw_root = task_root('corrosion_detection') / 'datasets_raw'
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (raw_root / sub).mkdir(parents=True, exist_ok=True)

print('Processed dataset :', datasets_dir('corrosion_detection'))
print('Optional raw input  :', raw_root)
print('Expected YOLO layout:')
print('  datasets_raw/images/train  +  datasets_raw/labels/train')
print('  datasets_raw/images/val    +  datasets_raw/labels/val')
print('Classes: fair (0), poor (1), severe (2)')

In [ ]:
# @title 4. Preprocess (auto-orient, stretch 640, CLAHE, class remap, 3x train copies)
from shared.preprocess_corrosion import preprocess_corrosion_dataset
from shared.paths import datasets_dir, task_root

RAW_DIR = task_root('corrosion_detection') / 'datasets_raw'
USE_RAW = any((RAW_DIR / 'images' / 'train').glob('*'))

yaml_path = preprocess_corrosion_dataset(
    raw_dir=RAW_DIR if USE_RAW else None,
    clear_existing=True,
)
print('Wrote dataset yaml:', yaml_path)
if USE_RAW:
    print('Raw folder used:', RAW_DIR)
else:
    print('In-place reprocess of:', datasets_dir('corrosion_detection'))

In [ ]:
# @title 5. Train YOLOv8 (configured)
from shared.trainer import train_corrosion

best_v8 = train_corrosion('yolov8', batch=8 if IN_COLAB else 8)
print('Done:', best_v8)

In [ ]:
# @title 6. Train YOLOv11 (configured)
from shared.trainer import train_corrosion

best_v11 = train_corrosion('yolov11', batch=16 if IN_COLAB else 8)
print('Done:', best_v11)

In [ ]:
# @title 7. Train YOLOv12 (configured)
from shared.trainer import train_corrosion

best_v12 = train_corrosion('yolov12', batch=16 if IN_COLAB else 8)
print('Done:', best_v12)

## Step-by-step training guide

### A. Prepare your data

**Option 1 — Roboflow export (recommended)**

1. Export from Roboflow as **YOLOv8 Segmentation**.
2. Match these Roboflow settings if possible:
   - Auto-Orient: on
   - Resize: Stretch 640×640
   - Auto-Adjust Contrast: Adaptive Equalization
   - Modify Classes: 2 remapped, 3 dropped
   - Augmentations: 3 outputs, flips, ±15° rotation, ±25% exposure, blur ≤2.5 px, noise ≤10%
3. Unzip into `corrosion_detection/datasets_raw/` with this layout:

```
datasets_raw/
  images/train/
  images/val/
  labels/train/
  labels/val/
```

**Option 2 — Already in YOLO layout**

Place files directly under `corrosion_detection/datasets/images/{train,val}` and `labels/{train,val}`. Cell 4 will reprocess them in place.

### B. Adjust class mapping (if needed)

Open `shared/corrosion_config.py` and edit:

- `CORROSION_CLASS_REMAP` — map old class ids to fair/poor/severe
- `CORROSION_DROP_CLASS_IDS` — ids to ignore

Default output classes: **fair (0), poor (1), severe (2)**.

### C. Run the notebook

| Cell | Action |
|---|---|
| 1 | Install deps + set repo path |
| 2 | Print active configuration |
| 3 | Create folder structure |
| 4 | Preprocess images (orient, stretch, CLAHE, remap, 3 train copies) |
| 5–7 | Train YOLOv8 / v11 / v12 (pick one or run all) |

### D. After training

- Weights: `corrosion_detection/runs/<model>/corrosion_<model>_seg/weights/best.pt`
- Metrics/plots: same run folder
- Download `best.pt` from Colab: Files panel → right-click → Download

### E. Notes

- **3 outputs per example:** offline copies use photometric aug (exposure/blur/noise). Flips and rotation are applied during YOLO training with correct mask transforms.
- **Batch size:** lower to `8` or `4` if Colab runs out of GPU memory.
- **Resume training:** call `train_corrosion('yolov8', resume=True)`.